# Rx PD Model - Colab Setup

## Setup Instructions:
1. Click the folder icon on the left to open the file browser
2. Upload your data files:
   - `Lending_default_train.zip`
   - `Lending_default_holdout.zip`
   - OR upload extracted CSV files directly
3. Run the cells below in order

In [ ]:
# Install required libraries
!pip install -q pandas numpy scikit-learn optbinning probatus shap matplotlib seaborn statsmodels -U

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import zipfile
import os
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, HalvingGridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score, \
    classification_report, confusion_matrix, mean_squared_error, balanced_accuracy_score, roc_curve, auc
from probatus.feature_elimination import ShapRFECV
from optbinning import OptimalBinning
from statsmodels.stats.outliers_influence import variance_inflation_factor
import shap
shap.initjs()

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully!')

## Step 1: Load and Extract Data

In [ ]:
# Check what files are in the current directory
print('Files in current directory:')
for file in os.listdir('.'):
    if file.endswith(('.csv', '.zip')):
        print(f'  - {file}')

In [ ]:
# Extract zip files if they exist
zip_files = ['Lending_default_train.zip', 'Lending_default_holdout.zip']

for zip_file in zip_files:
    if os.path.exists(zip_file):
        print(f'Extracting {zip_file}...')
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall()
        print(f'  ✓ Extracted')
    else:
        print(f'✗ {zip_file} not found. Please upload the zip file.')

# List available CSV files
csv_files = [f for f in os.listdir('.') if f.endswith('.csv') and not f.startswith('.')]
print(f'\nAvailable CSV files: {len(csv_files)}')
for f in sorted(csv_files):
    print(f'  - {f}')

## Step 2: Load Data

In [ ]:
# Load training data
print('Loading training data...')
df_train_tx = pd.read_csv('Lending_default_train_tx.csv')
df_train_acc = pd.read_csv('Lending_default_train_account.csv')
df_train_label = pd.read_csv('Lending_default_train_label.csv')

# Load holdout data
print('Loading holdout data...')
df_holdout_tx = pd.read_csv('Lending_default_holdout_tx.csv')
df_holdout_acc = pd.read_csv('Lending_default_holdout_account.csv')

# Remove unnamed index columns
df_train_tx = df_train_tx.loc[:, ~df_train_tx.columns.str.contains('^Unnamed')]
df_train_acc = df_train_acc.loc[:, ~df_train_acc.columns.str.contains('^Unnamed')]
df_train_label = df_train_label.loc[:, ~df_train_label.columns.str.contains('^Unnamed')]
df_holdout_tx = df_holdout_tx.loc[:, ~df_holdout_tx.columns.str.contains('^Unnamed')]
df_holdout_acc = df_holdout_acc.loc[:, ~df_holdout_acc.columns.str.contains('^Unnamed')]

print('✓ Data loaded successfully!')

## Step 3: Data Overview

In [ ]:
def check_missing(data):
    """Display missing values summary"""
    missing = data.isnull().sum().sort_values(ascending=False)
    percent = round((data.isnull().sum()/len(data)*100), 2).sort_values(ascending=False)
    missing_data = pd.concat([missing, percent], axis=1, keys=['Count', 'Percent'])
    return missing_data[missing_data['Count'] > 0]

print('='*80)
print('TRAINING TRANSACTION DATA')
print('='*80)
print(f'Shape: {df_train_tx.shape}')
print(f'Columns: {list(df_train_tx.columns)}')
print(f'\nFirst 5 rows:')
print(df_train_tx.head())

print('\n' + '='*80)
print('TRAINING ACCOUNT DATA')
print('='*80)
print(f'Shape: {df_train_acc.shape}')
print(f'Columns: {list(df_train_acc.columns)}')
print(f'\nFirst 5 rows:')
print(df_train_acc.head())
print(f'\nMissing values:')
print(check_missing(df_train_acc))

print('\n' + '='*80)
print('TRAINING LABEL DATA')
print('='*80)
print(f'Shape: {df_train_label.shape}')
print(f'\nLabel Distribution:')
print(df_train_label['loan_default'].value_counts())
event_rate = round(df_train_label['loan_default'].mean()*100, 2)
print(f'\nEvent Rate: {event_rate}%')

## Step 4: Prepare Modeling Dataset

In [ ]:
# Assuming you have already prepared a modeling dataset with features
# If not, merge the data first

# Create combined training dataset
df_train_modeling = df_train_tx.copy()  # Start with transaction features
df_train_modeling = pd.merge(df_train_modeling, df_train_acc, on='Restaurant_ID', how='inner')
df_train_modeling = pd.merge(df_train_modeling, df_train_label, on='Restaurant_ID', how='inner')

# Create combined holdout dataset
df_holdout_modeling = df_holdout_tx.copy()
df_holdout_modeling = pd.merge(df_holdout_modeling, df_holdout_acc, on='Restaurant_ID', how='inner')

print(f'Training dataset shape: {df_train_modeling.shape}')
print(f'Holdout dataset shape: {df_holdout_modeling.shape}')
print(f'\nTraining columns: {list(df_train_modeling.columns)}')

## Step 5: 80/20 Train-Test Split

In [ ]:
# Prepare data for modeling
data = df_train_modeling.copy()
y = data['loan_default']  # Target variable
remove_list = ['loan_default', 'Restaurant_ID', 'Tx_date']  # Remove target, ID, and dates
X = data.drop(columns=[col for col in remove_list if col in data.columns])

# 80/20 train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20,  # 20% for test
    train_size=0.80,  # 80% for train
    stratify=y,       # Maintain event rate in both sets
    random_state=42   # For reproducibility
)

# Create dataframes with target for later use
df_train = pd.merge(X_train, y_train, left_index=True, right_index=True, how='inner')
df_test = pd.merge(X_test, y_test, left_index=True, right_index=True, how='inner')

# Verify split
print('='*80)
print('TRAIN-TEST SPLIT SUMMARY')
print('='*80)
print(f'Total samples: {len(X):,}')
print(f'Training set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test set: {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nTarget variable distribution:')
print(f'  Training set event rate: {y_train.mean()*100:.2f}%')
print(f'  Test set event rate: {y_test.mean()*100:.2f}%')
print(f'  Overall event rate: {y.mean()*100:.2f}%')
print(f'\nDataframe shapes:')
print(f'  df_train: {df_train.shape}')
print(f'  df_test: {df_test.shape}')
print(f'\nNumber of features: {len(X.columns)}')
print(f'Features: {list(X.columns[:10])}... (showing first 10)')

## Step 6: Data Exploration

In [ ]:
# Data types and missing values
print('='*80)
print('TRAINING DATA - DATA QUALITY')
print('='*80)
print(f'\nData types:')
print(df_train.dtypes.value_counts())

print(f'\nMissing values:')
missing = check_missing(df_train)
if len(missing) > 0:
    print(missing)
else:
    print('No missing values ✓')

# Summary statistics
print(f'\nNumerical features summary:')
print(df_train.describe().round(2))

## Step 7: Feature Engineering & WoE Transformation (Optional)

In [ ]:
# Create feature dictionary for binning
feature_list = [col for col in X.columns if col != 'Restaurant_ID']
feature_dict = {}

for feature in feature_list:
    if X[feature].dtype in ['float64', 'int64']:
        feature_dict[feature] = 'numerical'
    else:
        feature_dict[feature] = 'categorical'

# Sort by type (categorical first)
feature_dict = dict(sorted(feature_dict.items(), key=lambda item: item[1]))

print(f'Feature Summary:')
print(f'  Numerical: {sum(1 for v in feature_dict.values() if v == "numerical")}')
print(f'  Categorical: {sum(1 for v in feature_dict.values() if v == "categorical")}')
print(f'  Total: {len(feature_dict)}')

## Step 8: Model Development (Logistic Regression)

In [ ]:
# Simple Logistic Regression baseline (without WoE)
print('Training Logistic Regression model...')

clf_lr = LogisticRegression(n_jobs=-1, random_state=42, max_iter=1000)
clf_lr.fit(X_train, y_train)

# Generate predictions
df_train['pred_lr'] = clf_lr.predict_proba(X_train)[:, 1]
df_test['pred_lr'] = clf_lr.predict_proba(X_test)[:, 1]

print('✓ Model trained')

# Evaluate
train_auc = roc_auc_score(y_train, df_train['pred_lr'])
test_auc = roc_auc_score(y_test, df_test['pred_lr'])

print(f'\nLogistic Regression Performance:')
print(f'  Training AUC: {train_auc:.4f}')
print(f'  Test AUC: {test_auc:.4f}')
print(f'  Difference: {abs(train_auc - test_auc):.4f}')

## Step 9: Model Development (Gradient Boosting)

In [ ]:
# Gradient Boosting Classifier
print('Training Gradient Boosting model...')

clf_gb = GradientBoostingClassifier(
    n_estimators=100,
    max_leaf_nodes=10,
    learning_rate=0.1,
    min_samples_leaf=0.05,
    random_state=42,
    verbose=0
)
clf_gb.fit(X_train, y_train)

# Generate predictions
df_train['pred_gb'] = clf_gb.predict_proba(X_train)[:, 1]
df_test['pred_gb'] = clf_gb.predict_proba(X_test)[:, 1]

print('✓ Model trained')

# Evaluate
train_auc_gb = roc_auc_score(y_train, df_train['pred_gb'])
test_auc_gb = roc_auc_score(y_test, df_test['pred_gb'])

print(f'\nGradient Boosting Performance:')
print(f'  Training AUC: {train_auc_gb:.4f}')
print(f'  Test AUC: {test_auc_gb:.4f}')
print(f'  Difference: {abs(train_auc_gb - test_auc_gb):.4f}')

## Step 10: Model Comparison

In [ ]:
# Compare models
print('='*80)
print('MODEL COMPARISON')
print('='*80)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Gradient Boosting'],
    'Train AUC': [train_auc, train_auc_gb],
    'Test AUC': [test_auc, test_auc_gb],
    'Overfit Gap': [abs(train_auc - test_auc), abs(train_auc_gb - test_auc_gb)]
})

print(comparison.round(4).to_string(index=False))

# Select best model
best_model_name = 'Gradient Boosting' if test_auc_gb > test_auc else 'Logistic Regression'
best_auc = max(test_auc, test_auc_gb)
print(f'\n✓ Best Model: {best_model_name} (Test AUC: {best_auc:.4f})')

## Step 11: Time Series Calibration - Actual vs Predicted by Period

In [ ]:
def plot_actual_vs_predicted_by_period(df, pred_col, actual_col, n_periods=10):
    """
    Plot actual vs predicted default rates over time/periods
    """
    # Create periods (deciles)
    df_temp = df.copy()
    df_temp['period'] = pd.qcut(range(len(df_temp)), q=n_periods, 
                                 labels=[f'Period_{i+1}' for i in range(n_periods)])
    
    # Calculate rates by period
    time_series = df_temp.groupby('period').agg({
        actual_col: 'mean',
        pred_col: 'mean'
    }).reset_index()
    
    time_series[actual_col] = time_series[actual_col] * 100
    time_series[pred_col] = time_series[pred_col] * 100
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = range(len(time_series))
    ax.plot(x, time_series[actual_col], marker='o', linewidth=2.5, 
            markersize=8, label='Actual Default Rate', color='#d62728')
    ax.plot(x, time_series[pred_col], marker='s', linewidth=2.5, 
            markersize=8, label='Predicted Default Rate', color='#1f77b4')
    
    ax.set_xlabel('Period', fontsize=12, fontweight='bold')
    ax.set_ylabel('Default Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Actual vs Predicted Default Rates by Period ({pred_col})', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(time_series['period'], rotation=45, ha='right')
    ax.legend(fontsize=11, loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Metrics
    mae = abs(time_series[actual_col] - time_series[pred_col]).mean()
    max_dev = abs(time_series[actual_col] - time_series[pred_col]).max()
    
    print(f'\nCalibration Metrics ({pred_col}):')
    print(f'  MAE (Mean Absolute Error): {mae:.2f}%')
    print(f'  Max Deviation: {max_dev:.2f}%')
    print(f'\nPeriod Summary:')
    print(time_series.round(2).to_string(index=False))
    
    return time_series

# Plot for both models
print('='*80)
print('TRAINING DATA: Actual vs Predicted by Period')
print('='*80)
train_ts = plot_actual_vs_predicted_by_period(df_train, 'pred_gb', 'loan_default')

print('\n' + '='*80)
print('TEST DATA: Actual vs Predicted by Period')
print('='*80)
test_ts = plot_actual_vs_predicted_by_period(df_test, 'pred_gb', 'loan_default')

## Step 12: Score Holdout Data

In [ ]:
# Prepare holdout features
X_holdout = df_holdout_modeling.drop(
    columns=[col for col in ['Restaurant_ID', 'Tx_date'] if col in df_holdout_modeling.columns]
)

# Ensure same features as training
for col in X_train.columns:
    if col not in X_holdout.columns:
        X_holdout[col] = 0  # Add missing columns with 0

X_holdout = X_holdout[X_train.columns]  # Match column order

# Score holdout using best model
if test_auc_gb > test_auc:
    holdout_pred = clf_gb.predict_proba(X_holdout)[:, 1]
    model_used = 'Gradient Boosting'
else:
    holdout_pred = clf_lr.predict_proba(X_holdout)[:, 1]
    model_used = 'Logistic Regression'

# Create output
holdout_results = pd.DataFrame({
    'Restaurant_ID': df_holdout_modeling['Restaurant_ID'].values,
    'Predicted_Default_Probability': np.round(holdout_pred, 4),
    'Predicted_Default_Score': np.round(holdout_pred * 100, 2)
})

print(f'='*80)
print(f'HOLDOUT SCORING - {model_used}')
print(f'='*80)
print(f'Total records scored: {len(holdout_results):,}')
print(f'\nPrediction Statistics:')
print(holdout_results['Predicted_Default_Probability'].describe().round(4))
print(f'\nFirst 10 predictions:')
print(holdout_results.head(10).to_string(index=False))

# Save results
holdout_results.to_csv('holdout_predictions.csv', index=False)
print(f'\n✓ Results saved to holdout_predictions.csv')

## Step 13: Download Results

In [ ]:
# In Colab, files are saved to the session directory
# Use the file browser on the left to download:
# - holdout_predictions.csv

print('Files created:')
print('  - holdout_predictions.csv')
print('\nTo download in Colab:')
print('  1. Click the folder icon on the left')
print('  2. Right-click the file')
print('  3. Select "Download"')

# Alternative: use this to download programmatically
from google.colab import files
# files.download('holdout_predictions.csv')

## Summary

In [ ]:
print('='*80)
print('MODELING SUMMARY')
print('='*80)
print(f'\nData Split:')
print(f'  Training samples: {len(df_train):,}')
print(f'  Test samples: {len(df_test):,}')
print(f'  Holdout samples: {len(holdout_results):,}')
print(f'\nBest Model: {best_model_name}')
print(f'  Test AUC: {best_auc:.4f}')
print(f'\nNext Steps:')
print(f'  1. Download holdout_predictions.csv')
print(f'  2. Review predictions and performance metrics')
print(f'  3. Consider further feature engineering if needed')
print(f'  4. Deploy model for production scoring')